In [1]:
from scpviz import pAnnData as pAnnData
from scpviz import plotting as scplt
from scpviz import utils as scutils

# Tutorial 1: Importing Data

This tutorial shows how to import **DIA-NN** or **Proteome Discoverer (PD)** outputs into a `pAnnData` object.

`scpviz` currently supports:

- **Proteome Discoverer** (tested on versions 2.5 and 3.2)
- **DIA-NN** (tested on versions 1.8.1, 2.0, and 2.1)

`pAnnData` objects integrate **protein** and **peptide** level data:

- **DIA-NN** reports contain everything required.
- **PD** protein exports are required; peptide exports are optional but recommended for peptide-level filtering and analysis.

## Encoding metadata

It’s important to encode metadata about samples (e.g., knockdown vs scrambled control) for downstream grouping, filtering, and visualization.


### DIA-NN Encoding
In DIA-NN, sample metadata should be encoded in the raw filenames. For example:
`20251106_Caltech-Marion_Astral_25min_Aur25cm_KD-01.raw`

Split by `_`, the tokens become:

- date: `20251106`
- user: `Caltech-Marion`
- mass spectrometer: `Astral`
- gradient length: `25min`
- column: `Aur25cm`
- sample condition + replicate: `KD-01`

The `.raw` extension is automatically dropped during metadata parsing.

### PD encoding
In Proteome Discoverer, metadata is encoded via **categorical variables** in the study design.
Typical steps:

1. Create a categorical variable (e.g. `sample_condition`) on the study page.
2. Add possible values (e.g. `control`, `kd`).
3. Assign values in the **Samples** tab.

Note:
- `scpviz` automatically detects and assigns a delimiter based on the most frequent character. If that fails, specify your own via the `delimiter` argument during import.

- During import, `scpviz` checks filename token lengths and suggests `.obs` column names if they are uniform. If filenames contain multiple token lengths, they are grouped as `parsingType = "10-tokens"`, `"6-tokens"`, etc.

## Loading DIA-NN reports
For DIA-NN, the `report.parquet` file is all you need. It includes peptide-level detail for each file, allowing `scpviz` to build the protein–peptide matrices.

In [ ]:
# warning: this file is 200+ MB large, and may take a while to download and load
!wget -q https://github.com/gnaprs/scpviz/releases/download/v0.5.2-alpha/diann_report.parquet

In [ ]:
from scpviz import pAnnData as pAnnData

obs_columns = ["user", "date", "ms", "acquisition", "faims", "column", "gradient", "amount", "region", "rep"]
pdata = pAnnData.import_data(
    source_type="diann",
    report_file="diann_report.parquet",
    obs_columns=obs_columns,
)

🧭 [USER] Importing data of type [diann]
--------------------------
Starting import [DIA-NN]

Source file: ../../tests/test_diann.parquet
Number of files: 12
Proteins: 2251
Peptides: 7688

     ℹ️ [INFO] Using sample-specific q-values for 'prot' significance annotation.
     ℹ️ [INFO] Using sample-specific q-values for 'pep' significance annotation.

ℹ️ RS matrix: (2251, 7688) (proteins × peptides), sparsity: 99.95%
   - Proteins with ≥2 *unique* linked peptides: 1217/2251
   - Peptides linked to ≥2 proteins: 158/7688
   - Mean peptides per protein: 3.53
   - Mean proteins per peptide: 1.03
     ✅ [OK] pAnnData object is valid.
     ✅ [OK] Import complete. Use `print(pdata)` to view the object.
--------------------------


## Loading Proteome Discoverer (PD) reports
### Export requirements

For PD, we recommend modifying the export layout to include **raw abundances** (typically `Abundances`). The default layout is usually **scaled abundance**, which is not ideal for quantitative workflows. For convenience, here is a custom layout file that you can load into PD: [pd_scpviz_layout.pdLayout](https://github.com/gnaprs/scpviz/raw/main/docs/assets/pd_scpviz_layout.pdLayout)

Proteome Discoverer allows export of specific tabs. We recommend exporting as **tab-delimited text** (Excel is supported but much larger in file size and thus slower to load).

Make sure to export **at minimum** the following tabs:

- **Protein**
- (optional, but recommended) **Peptide Groups** (PD2.5) or **Peptide Sequence Groups** (PD3.2)

### Import for PD3.2

In [ ]:
!wget -q https://github.com/gnaprs/scpviz/raw/main/docs/assets/pd32_Proteins.txt
!wget -q https://github.com/gnaprs/scpviz/raw/main/docs/assets/pd32_PeptideSequenceGroups.txt

In [5]:
prot_file_path = "pd32_Proteins.txt"
pep_file_path = "pd32_PeptideSequenceGroups.txt"
obs_columns = ['sample', 'cellline', 'treatment', 'condition', 'day']

pdata = pAnnData.import_data(
    source_type="pd",
    prot_file=prot_file_path,
    pep_file=pep_file_path,
    obs_columns=obs_columns,
)

🧭 [USER] Importing data of type [pd]
--------------------------
Starting import [Proteome Discoverer]

Source file: ../assets/pd32_Proteins.txt / ../assets/pd32_PeptideSequenceGroups.txt
Number of files: 12
Proteins: 10393
Peptides: 167114

ℹ️ Using 'Modifications in Master Proteins' for modification annotation.
     ⚠️ [WARN] Master proteins in the peptide matrix do not match proteins in the protein data, please check if files correspond to the same data.
     ℹ️ [INFO] If using PD3.2, this is a known issue due to changed protein grouping rules.

ℹ️ 73 proteins with missing gene names.
     🌐 [API] Querying UniProt for batch 1/1 (73 proteins) [fields: accession, gene_primary]
     ✅ Retrieved UniProt metadata for 71 entries.
     ✅ [OK] Recovered 70 gene name(s) from UniProt. Genes found:
         PCM1, HDLBP, GBF1, WDR36, DDX27, MTCL2, ATAD3A, SLC4A1AP, COG5, DBT...
     ⚠️ [WARN] 3 gene name(s) still missing. Assigned as 'UNKNOWN_<accession>' for:
         A0A0B4J2D5, Q6ZSR9, A9Z1Z3

### Import for PD2.5

In [ ]:
!wget -q https://github.com/gnaprs/scpviz/raw/main/docs/assets/pd25_Proteins.txt
!wget -q https://github.com/gnaprs/scpviz/raw/main/docs/assets/pd25_PeptideGroups.txt

In [ ]:
prot_file_path = "pd25_Proteins.txt"
pep_file_path = "pd25_PeptideGroups.txt"
obs_columns = ['sample', 'cellline', 'condition']

pdata = pAnnData.import_data(
    source_type="pd",
    prot_file=prot_file_path,
    pep_file=pep_file_path,
    obs_columns=obs_columns,
)

🧭 [USER] Importing data of type [pd]
--------------------------
Starting import [Proteome Discoverer]

Source file: ../assets/pd25_Proteins.txt / ../assets/pd25_PeptideGroups.txt
Number of files: 12
Proteins: 4988
Peptides: 30920

ℹ️ Using 'Modifications' for modification annotation.

ℹ️ 25 proteins with missing gene names.
     🌐 [API] Querying UniProt for batch 1/1 (25 proteins) [fields: accession, gene_primary]
     ✅ Retrieved UniProt metadata for 25 entries.
     ✅ [OK] Recovered 24 gene name(s) from UniProt. Genes found:
         TUFM, HDLBP, AMPD2, MYG1, HSD17B11, PCM1, NEFH, OXA1L, TRMT5, SLC4A1AP...
     ⚠️ [WARN] 1 gene name(s) still missing. Assigned as 'UNKNOWN_<accession>' for:
         Q6ZSR9
     💡 Tip: You can update these using `pdata.update_identifier_maps({'GENE': 'ACCESSION'}, on='protein', direction='reverse', overwrite=True)`

ℹ️ Removed 64 empty proteins (all-NaN or all-zero). Proteins: P49448, P53675, Q8IZP2, Q15173, O43402, P40123, O95954, O75382, Q9NYQ6, Q14BN

Note:
- PD uses **global FDR** (unlike DIA-NN, which provides per-precursor / per-protein FDR). This does not affect import but may influence downstream filtering decisions.

### Gene name recovery

During import, `scpviz` automatically checks for proteins with missing gene names and queries **UniProt** to recover them.

Proteins without gene names after UniProt lookup are assigned as `UNKNOWN_<accession>` and can be manually updated later if needed using `pdata.update_identifier_maps()`.

## Mapping accessions and peptides

After import, `pAnnData` stores three linked objects:

- **`.prot`** — protein-level abundances (rows = samples, columns = accessions)
- **`.pep`** — peptide-level abundances (rows = samples, columns = peptide IDs)
- **`.rs`** — sparse protein × peptide relational matrix built during import

Use the RS matrix to translate between protein accessions and the peptides observed in your dataset. Both directions accept flexible inputs (accessions or gene names; peptide IDs or amino-acid strings).

In [6]:
from scpviz import utils as scutils

### Accession → peptides

`get_peptides_for_accessions()` returns a DataFrame with columns `accession`, `peptide_id`, and `sequence`.

In [9]:
df = scutils.get_peptides_for_accessions(pdata, ["Q9CZW5"])

# by gene name
df = scutils.get_peptides_for_accessions(pdata, ["Tomm70"])

df.head()

,accession,peptide_id,sequence
0,Q9CZW5,AAAFEQLQK2,AAAFEQLQK2
1,Q9CZW5,GFEEIIK2,GFEEIIK2
2,Q9CZW5,GLLQLQWK2,GLLQLQWK2


By default, `sequence` is taken from `.pep.var_names` (`sequence_from="index"`):

| Source | `.pep.var_names` | Default `sequence` column |
|:-------|:-----------------|:--------------------------|
| **Proteome Discoverer** | Annotated sequence (+ optional modifications) | Same as `peptide_id` |
| **DIA-NN** | `Precursor.Id` | Same as `peptide_id` (precursor ID, not amino acids) |

For DIA-NN, request amino-acid strings explicitly:

In [12]:
df = scutils.get_peptides_for_accessions(
    pdata,
    ["Tomm70"],
    sequence_from="Stripped.Sequence",
)

df

,accession,peptide_id,sequence
0,Q9CZW5,AAAFEQLQK2,AAAFEQLQK
1,Q9CZW5,GFEEIIK2,GFEEIIK
2,Q9CZW5,GLLQLQWK2,GLLQLQWK


Unmatched accessions or genes are skipped with a warning; remaining matches are still returned.

### Peptide → accessions

`get_accessions_for_peptides()` returns `peptide_id`, `accession`, and `sequence`. Inputs may be `.pep.var_names` or amino-acid strings (matched against `Stripped.Sequence`, `Modified.Sequence`, or `Annotated Sequence`).

In [13]:
pep_id = pdata.pep.var_names[0]
df = scutils.get_accessions_for_peptides(pdata, [pep_id])

df

,peptide_id,accession,sequence
0,AAAELLQSQGSQAGGSQTLK3,Q9R1T4,AAAELLQSQGSQAGGSQTLK3


In [14]:
seq = pdata.pep.var["Stripped.Sequence"].iloc[0]
df = scutils.get_accessions_for_peptides(
    pdata,
    [seq],
    sequence_from="Stripped.Sequence",
)

df

,peptide_id,accession,sequence
0,AAAELLQSQGSQAGGSQTLK3,Q9R1T4,AAAELLQSQGSQAGGSQTLK


Shared peptides linked to multiple proteins produce **one row per accession**. A single sequence string can also match **multiple precursor IDs** when several precursors share the same stripped sequence.

### Round-trip check

In [15]:
acc = pdata.prot.var_names[0]
peps = scutils.get_peptides_for_accessions(pdata, [acc])
prots = scutils.get_accessions_for_peptides(pdata, [peps.iloc[0]["peptide_id"]])
assert acc in prots["accession"].values

!!! note
    These functions require protein, peptide, and RS data (peptide import must be included). They inspect the RS matrix without modifying `pdata`. To **filter** by peptide support instead, see [`filter_rs()`](filtering.md#filter_rs) in the filtering tutorial.

## Metadata parsing

Sample metadata (columns in `.obs`) can be inferred directly from filenames:

In [9]:
pdata.summary

,sample,cellline,condition,protein_quant,protein_count,protein_abundance_sum,mbr_count,high_count,peptide_quant,peptide_count,peptide_abundance_sum,unique_pep2_protein_count
F4,Sample,AS,kd,0.959789,4726,6.543988e+10,1322,3400,0.916451,28278,6.543988e+10,3714
F8,Sample,AS,kd,0.953899,4697,5.114338e+10,1519,3170,0.906112,27959,5.114338e+10,3710
F12,Sample,AS,kd,0.951868,4687,5.662568e+10,1396,3283,0.900311,27780,5.662568e+10,3713
F2,Sample,AS,sc,0.948619,4671,5.097928e+10,1565,3099,0.897265,27686,5.097928e+10,3709
F6,Sample,AS,sc,0.958773,4721,5.664297e+10,1565,3151,0.902742,27855,5.664297e+10,3718
F10,Sample,AS,sc,0.956336,4709,6.136897e+10,1392,3316,0.897232,27685,6.136897e+10,3718
F3,Sample,BE,kd,0.961007,4732,5.836572e+10,1352,3381,0.927145,28608,5.836572e+10,3724
F7,Sample,BE,kd,0.959383,4724,5.590369e+10,1317,3408,0.929284,28674,5.590369e+10,3719
F11,Sample,BE,kd,0.961617,4735,8.650390e+10,1202,3542,0.925493,28557,8.650390e+10,3717
F1,Sample,BE,sc,0.963444,4744,7.587834e+10,1329,3416,0.925039,28543,7.587834e+10,3720


Updates to `.summary` are automatically pushed to `.prot.obs` and `.pep.obs` (if available). If `scpviz` can’t infer whether a change is intentional, you’ll be prompted to run `pdata.update_summary()`.

### When filenames follow a single format

If all filenames share the same number of tokens, `scpviz` will suggest `obs_columns` from the first filename and ask you to confirm or edit them. This is common for PD exports when filenames encode basic sample info.

In [11]:
prot_file_path = "pd32_Proteins.txt"
pep_file_path = "pd32_PeptideSequenceGroups.txt"

pdata = pAnnData.import_data(
    source_type="pd",
    prot_file=prot_file_path,
    pep_file=pep_file_path,
)

🧭 [USER] Importing data of type [pd]
      Auto-detecting ',' as delimiter from first filename.
ℹ️ Filenames are uniform. Using `suggest_obs_columns()` to recommend obs_columns...

From filename: Sample, AS, RA, kd, d7
Suggested .obs columns:
  unknown??                 : Sample
  unknown??                 : AS
  unknown??                 : RA
  condition                 : kd
  unknown??                 : d7
Unrecognized token(s): ['Sample', 'AS', 'RA', 'd7']
Please manually label these.

ℹ️ Suggested obs:
obs_columns = ['<Sample?>', '<AS?>', '<RA?>', 'condition', '<d7?>']
     ⚠️ [WARN] Please review the suggested `obs_columns` above.
   → If acceptable, rerun `import_data(..., obs_columns=...)` with this list.



In this case, you should fill in `obs_columns` with meaningful labels and rerun the import. For example:

In [ ]:
obs_columns = ["sample", "cellline", "treatment", "condition", "day"]

Then, we can import our data:

In [ ]:
pdata = pAnnData.import_data(
        source_type="pd",
        prot_file=prot_file_path,
        pep_file=pep_file_path,
        obs_columns=obs_columns,
)

🧭 [USER] Importing data of type [pd]
--------------------------
Starting import [Proteome Discoverer]

Source file: ../assets/pd32_Proteins.txt / ../assets/pd32_PeptideSequenceGroups.txt
Number of files: 12
Proteins: 10393
Peptides: 167114

ℹ️ Using 'Modifications in Master Proteins' for modification annotation.
     ⚠️ [WARN] Master proteins in the peptide matrix do not match proteins in the protein data, please check if files correspond to the same data.
     ℹ️ [INFO] If using PD3.2, this is a known issue due to changed protein grouping rules.

ℹ️ 73 proteins with missing gene names.
     🌐 [API] Querying UniProt for batch 1/1 (73 proteins) [fields: accession, gene_primary]
     ✅ Retrieved UniProt metadata for 71 entries.
     ✅ [OK] Recovered 70 gene name(s) from UniProt. Genes found:
         PCM1, HDLBP, GBF1, WDR36, DDX27, MTCL2, ATAD3A, SLC4A1AP, COG5, DBT...
     ⚠️ [WARN] 3 gene name(s) still missing. Assigned as 'UNKNOWN_<accession>' for:
         A0A0B4J2D5, Q6ZSR9, A9Z1Z3

If filenames follow multiple formats, use `parse_filename_index` to handle different token lengths.

**Parse by token length:**

```python
pdata.summary = scutils.parse_filename_index(
    pdata.summary,
    obs_columns=["date", "acquisition", "sample_id", "size", "confirmation", "thickness", "type", "organism", "region", "well_position"],
    condition='parsingType == "10-tokens"', delimiter="-",
)

pdata.summary = scutils.parse_filename_index(
    pdata.summary,
    obs_columns=["date", "sample_id", "size", "thickness", "organism", "region"],
    condition='parsingType == "6-tokens"', delimiter="-",
)
```

**Parse all filenames:**

```python
pdata.summary = scutils.parse_filename_index(
    pdata.summary,
    obs_columns=["date", "acquisition", "size", "buffer", "well_position"],
)
```